# Split the mined rules into pairwise and complex

The pipeline writes one file, `results_CN.csv`, holding every rule it mined - the ones naming
two cell items and the longer ones naming three or more. That file is large, and every notebook
reading it would pay to parse the whole thing before throwing most of it away.

So it gets split once, here, into two files sitting beside it:

- `results_CN_pairwise.csv` - two items. What the pair-analysis and rule-space notebooks read.
- `results_CN_complex.csv` - three or more. What the complex-rule notebook reads.

Nothing is filtered here. Splitting only, plus one column counting how many FOVs each rule
appears in. The notebooks do their own filtering, so a rule left out here could never be
asked for later.

Run this once after a mining run, then the notebooks.

In [1]:
import os

import pandas as pd

import data_helper as dh

# ====================================================================
# Configuration
# ====================================================================

INPUT_FILENAME    = 'results_CN.csv'
PAIRWISE_FILENAME = dh.PAIRWISE_FILENAME
COMPLEX_FILENAME  = dh.COMPLEX_FILENAME

# Which run to split. dh.RESULT_CSV_PATH is the one every notebook reads; point this
# somewhere else to split a run the notebooks are not looking at.
RESULT_CSV_DIR = dh.RESULT_CSV_PATH

print(f'Run: {RESULT_CSV_DIR}')

Run: c:\Users\Owner\Documents\worktrees\script-rule-mining-wt-2\results\full_run\binary_CN_4_items_fixed_Epithelial\data


## Load

In [2]:
def load_rules(result_csv_dir=None, filename=INPUT_FILENAME):
    """Read the run's rules: the whole table, exactly as the pipeline wrote it."""
    path = os.path.join(result_csv_dir or RESULT_CSV_DIR, filename)
    rules = pd.read_csv(path)
    print(f'Loaded {len(rules)} rules from {path}')
    return rules

## Split them

By how many items the rule names, counting roles: `Paneth_CENTER + Paneth_NEIGHBOR -> CD4T` is
three items, so it is complex even though it only names two cell types.

In [3]:
def split_by_size(rules):
    """Two frames: the rules naming two items, and the rules naming three or more."""
    items = rules.apply(dh.count_items, axis=1)
    pairwise, complex_rules = rules[items == 2].copy(), rules[items > 2].copy()
    print(f'{len(pairwise)} pairwise, {len(complex_rules)} complex')
    return pairwise, complex_rules

## Check the split before writing anything

The library labels every rule itself, in `Rule_Type`, counting items the same way this notebook
does. So the two have to agree: `pairwise` must mean exactly the two-item rules, and nothing
else. If they ever disagree the two definitions have drifted apart, and the split would quietly
send rules to the wrong file - where the notebook reading that file would never think to look
for them. Better to stop here than to find out from a figure.

In [4]:
def check_rule_type_agrees(pairwise, complex_rules):
    """Stop unless the library's own Rule_Type says what this split says."""
    if 'Rule_Type' not in pairwise.columns:
        print('No Rule_Type column, nothing to check the split against.')
        return

    called_complex = pairwise[pairwise['Rule_Type'] != 'pairwise']
    called_pairwise = complex_rules[complex_rules['Rule_Type'] == 'pairwise']

    if len(called_complex) or len(called_pairwise):
        example = pd.concat([called_complex, called_pairwise]).head(5)
        raise AssertionError(
            f'Rule_Type disagrees with the split on {len(called_complex) + len(called_pairwise)} '
            f'rules: {len(called_complex)} two-item rules the library calls complex, '
            f'{len(called_pairwise)} longer rules it calls pairwise. Nothing was written.\n'
            f'{example[["FOV", "Antecedents", "Consequents", "Rule_Type"]].to_string()}'
        )

    shapes = complex_rules['Rule_Type'].value_counts().to_dict()
    print(f'Rule_Type agrees with the split. Complex shapes: {shapes}')

## How widespread each rule is

How many FOVs the same rule turned up in, which is what the pair-analysis notebook ranks by.
Counted over everything, before any notebook narrows the rules down, so the number means the
same thing whatever a notebook goes on to filter.

In [5]:
def add_rule_count(rules):
    """Add Rule_Count_Global: how many FOVs this exact rule appears in."""
    rules['Rule_Count_Global'] = (
        rules.groupby(['Antecedents', 'Consequents'])['FOV'].transform('nunique')
    )
    return rules

## Write the two files

In [6]:
def save(pairwise, complex_rules, result_csv_dir=None):
    """Write both files beside the one they came from."""
    directory = result_csv_dir or RESULT_CSV_DIR
    for frame, filename in ((pairwise, PAIRWISE_FILENAME), (complex_rules, COMPLEX_FILENAME)):
        path = os.path.join(directory, filename)
        frame.to_csv(path, index=False)
        print(f'Saved {len(frame)} rules to {path}')

## Run it

In [7]:
rules = load_rules()
pairwise, complex_rules = split_by_size(rules)

check_rule_type_agrees(pairwise, complex_rules)

save(add_rule_count(pairwise), add_rule_count(complex_rules))

Loaded 1190114 rules from c:\Users\Owner\Documents\worktrees\script-rule-mining-wt-2\results\full_run\binary_CN_4_items_fixed_Epithelial\data\results_CN.csv
16600 pairwise, 1173514 complex
Rule_Type agrees with the split. Complex shapes: {'ant-complex': 619827, 'both-complex': 391682, 'con-complex': 162005}
Saved 16600 rules to c:\Users\Owner\Documents\worktrees\script-rule-mining-wt-2\results\full_run\binary_CN_4_items_fixed_Epithelial\data\results_CN_pairwise.csv
Saved 1173514 rules to c:\Users\Owner\Documents\worktrees\script-rule-mining-wt-2\results\full_run\binary_CN_4_items_fixed_Epithelial\data\results_CN_complex.csv
